In [573]:
import os
import pdfplumber
import re
import pandas as pd
from tqdm import tqdm

pth = r'C:\Users\a.vorona\Desktop\Work\Sunward\SWT22J_SP_EN.pdf'
filename = os.path.basename(pth)          
filename = os.path.splitext(filename)[0]
print(f'Обработка файла: {filename}')

Обработка файла: SWT22J_SP_EN


In [574]:
def extract_tables_camelot(pdf_path: str, out_dir="tables", pages="9-77"):
    os.makedirs(out_dir, exist_ok=True)

    with pdfplumber.open(pdf_path) as pdf:
        for page_i in range(len(pdf.pages)):
            page = pdf.pages[page_i]
            text = page.extract_text() or ""

            with open(os.path.join(out_dir, f"txt_{page_i+1}.txt"),
                      "w", encoding="utf-8") as f:
                f.write(text)

In [575]:
def extract_descr(text: str) -> str:
    pos_index = text.find("POS")
    if pos_index == -1:
        return text  
    return text[:pos_index].strip() 


def split_eng_chinese(text):
    # Ищем первый китайский символ
    match = re.search(r'[\u4e00-\u9fff]', text)
    if match:
        idx = match.start()
        # ищем, есть ли перед китайским символом код, состоящий из латиницы и цифр
        code_match = re.search(r'[A-Za-z0-9/\-]+$', text[:idx])
        if code_match:
            code_start = code_match.start()
            english = text[:code_start].rstrip()
            chinese = text[code_start:].lstrip()
        else:
            english = text[:idx].rstrip()
            chinese = text[idx:].lstrip()
        return english, chinese
    else:
        return text, ""


def line_to_list(line: str):
    itog = []
    l = line.strip().split(' ')
    itog.append(l[1])
    itog.append(l[2])
    eng_ch = ' '.join(l[3:-2])
    eng, chi = split_eng_chinese(eng_ch)
    itog.append(eng)
    itog.append(chi)
    itog.append(l[-2])
    itog.append(l[-1])
    
    return itog

def join_ordered(series):
    vals = series.dropna().astype(str).str.strip()
    vals = [v for v in vals if v]
    seen = set()
    result = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            result.append(v)
    return "; ".join(result) if result else None


def merge_parts(df):
    df = df.copy()
    
    agg_map = {
        'VER': join_ordered,
        'NAME_EN': join_ordered,
        'NAME_CN': join_ordered,
        "QTY": "sum",
        "SUIT_NUM": join_ordered,
        'MODEL': join_ordered,
        "DESCRIPTION": join_ordered,
    }

    result = (
        df.groupby("CODE", as_index=False)
        .agg(agg_map)
    )
    return result

In [576]:
folder = os.getcwd()
pdf_files = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".pdf")]

for pth in pdf_files:

    filename = os.path.basename(pth)          
    filename = os.path.splitext(filename)[0]

    extract_tables_camelot(pth, filename)
    
    columns = ["CODE", "VER", "NAME_EN", "NAME_CN", "QTY", "SUIT_NUM", "MODEL", 'DESCRIPTION']
    df = pd.DataFrame(columns=columns)

    pattern = re.compile(r"[（(](.*?)[）)]")

    def extract_in_brackets(s):
        if isinstance(s, str):
            match = pattern.search(s)
            if match:
                return match.group(1)
        return None

    for file in os.listdir(filename):
        if not file.startswith("txt_") or not file.lower().endswith(".txt"):
            continue
        path = os.path.join(filename, file)
        with open(path, 'r', encoding='utf-8') as f:
            text = f.readlines()

        model = extract_in_brackets(text[0])
        if len(text) < 2:
            continue
        
        descr = extract_descr(text[2])

        if model is None or len(model) == 0:
            continue


        for i in text[3:]:
            if not i[0].isdigit():
                continue
            print(i)
            row = line_to_list(i)
            df.loc[len(df)] = row + [model] + [descr]

    df = pd.DataFrame(df)
    df['QTY'] = pd.to_numeric(df['QTY']).astype(int)
    df_merged = merge_parts(df)

    df_merged.to_excel(f"{filename}_модели.xlsx", index=False)

1 802U01050000 S1 FRAME FRONT COVER 车架前盖板 1 102-99999

2 701104010034 S1 BOLT 螺栓 53 102-99999

3 701506010003 S1 WASHER 垫圈 106 102-99999

4 701510010004 S1 WASHER 垫圈 153 102-99999

5 802U01010000 S2 CHASSIS WEIDING ASSEMBLY 底盘焊接总成 1 102-99999

6 802U01060000 S1 FRAME REAR COVER 车架后盖板 1 102-99999

7 750301000368 S1 WHEEL REDUCER IFT007T2B048A 轮边减速机IFT007T2B048A 4 102-99999

8 750123000169 S1 RIGHT FILLED TIRE ASSEMBIY 355/55D625 右填充轮胎总成355/55D625 2 102-99999

9 751103000320 S1 NUT 轮辋螺母M16X1.5-10.9-DKL 36 102-99999

10 750123000170 S1 LEFT FILLED TIRE ASSEMBIY 355/55D625 左填充轮胎总成355/55D625 2 102-99999

11 802U01020000 S2 STEERING AXLE INSTALLATION 转向桥安装 1 102-99999

12 302540034191 S1 CABLE BRACKET 1 过线支架1 2 102-99999

13 701104008027 S1 BOLT 螺栓 20 102-99999

14 701504008005 S1 WASHER 垫圈 272 102-99999

15 701510008003 S1 WASHER 垫圈 81 102-99999

16 320990001967 S1 PIPE CLAMP 管夹 2 102-99999

17 701406008004 S1 NUT 螺母 48 102-99999

18 802U01040000 S1 OIL CATCH BLOCK INSTALLATION 集油块安装件 1 102

In [577]:
folder = os.getcwd()
excels = [os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(".xlsx")]

dfs = []
for i, pth in enumerate(tqdm(excels)):
    df_i = pd.read_excel(pth)
    df_i = df_i.loc[:, ['CODE', 'NAME_EN', 'MODEL']]
    print(df_i.shape[0])
    dfs.append(df_i)

df_all = pd.concat(dfs, ignore_index=True)

100%|██████████| 2/2 [00:00<00:00, 13.10it/s]

786
762


In [578]:
def join_ordered_description2(series: pd.Series):
    all_parts = []

    for item in series.dropna():
        text = str(item)
        # Убираем пробелы и невидимые символы
        text = re.sub(r'\s+', ' ', text, flags=re.UNICODE).strip()
        if not text:
            continue

        # Разделяем по ; если есть
        parts = [p.strip() for p in text.split(";") if p.strip()]

        all_parts.extend(parts)

    seen = set()
    result = []

    for v in all_parts:
        # ключ для проверки уникальности
        key = v.lower()
        if key not in seen:
            seen.add(key)
            result.append(v)

    return "; ".join(result) if result else None




def join_ordered2(series: pd.Series):
    all_parts = []

    for item in series.dropna(): 
        parts = str(item).split(';') 
        for p in parts:
            p_clean = re.sub(r'\s+', '', p).upper()
            if p_clean:
                all_parts.append(p_clean)

    seen = set()
    result = []
    for v in all_parts:
        if v not in seen:
            seen.add(v)
            result.append(v)

    return "; ".join(result) if result else None


def merge_parts2(df):
    df = df.copy()

    agg_map = {
        "NAME_EN": join_ordered_description2,
        "MODEL": join_ordered2,
    }

    df_agg = df.groupby("CODE", as_index=False).agg(agg_map)

    return df_agg

In [579]:
df_merged = merge_parts2(df_all)

In [580]:
df_merged.to_excel('Sunward_models.xlsx', index=False)